In [2]:
from fastapi.testclient import TestClient  # Cliente para realizar pruebas a la API FastAPI
from pathlib import Path  # Manejo de rutas de archivos de forma multiplataforma
from api.main import app  # Importamos la aplicación FastAPI desde el módulo api.main
import pandas as pd  # Biblioteca para manipulación y análisis de datos
import numpy as np  # Biblioteca para operaciones numéricas y arrays
import traceback  # Para imprimir trazas de errores detalladas
import warnings  # Para gestionar advertencias durante la ejecución
import joblib  # Para cargar modelos guardados en formato pickle
import sys  # Acceso a variables y funciones del sistema
import os  # Interacción con el sistema operativo (rutas, variables de entorno)

# Ignoramos advertencias específicas de Pydantic para evitar ruido en la salida
warnings.filterwarnings("ignore", category=UserWarning, module="pydantic.*")

# Creamos el cliente de pruebas que simulará peticiones HTTP a nuestra API
client = TestClient(app)

def diagnosticar_modelo():
    """
    Función principal de diagnóstico que evalúa el comportamiento de la API
    y del modelo de predicción de alquileres.
    """
    print("=" * 70)
    print("🔍 DIAGNÓSTICO DE API - Predicción de Alquileres")
    print("=" * 70)
    
    # 1. Verificar health check
    print("\n1. VERIFICANDO ESTADO DE LA API...")
    health = client.get("/health")  # Realizamos petición GET al endpoint /health
    if health.status_code == 200:  # Verificamos que la respuesta sea exitosa
        health_data = health.json()  # Convertimos la respuesta a JSON
        print(f"   ✓ Health check: {health_data}")
        if not health_data.get("model_loaded"):  # Verificamos si el modelo está cargado
            print("   ✗ ERROR: El modelo no está cargado")
            return
    else:
        print(f"   ✗ Error en health check: {health.status_code}")
        return
    
    # 2. Probar diferentes combinaciones de entrada
    print("\n2. PROBANDO DIFERENTES ESCENARIOS...")
    
    # Definimos varios escenarios de prueba con diferentes valores
    escenarios = [
        {
            "nombre": "Mínimo posible",  # Valores mínimos típicos
            "datos": {
                "provincia": "Pichincha",
                "lugar": "Quito",
                "num_dormitorios": 1,
                "num_banos": 1,
                "area": 30,
                "num_garages": 0
            }
        },
        {
            "nombre": "Máximo posible",  # Valores máximos típicos
            "datos": {
                "provincia": "Guayas",
                "lugar": "Guayaquil",
                "num_dormitorios": 6,
                "num_banos": 5,
                "area": 500,
                "num_garages": 4
            }
        },
        {
            "nombre": "Valores medios",  # Valores promedio
            "datos": {
                "provincia": "Azuay",
                "lugar": "Cuenca",
                "num_dormitorios": 3,
                "num_banos": 2,
                "area": 150,
                "num_garages": 2
            }
        },
        {
            "nombre": "Provincia diferente",  # Combinación con provincia diferente
            "datos": {
                "provincia": "Manabí",
                "lugar": "Manta",
                "num_dormitorios": 2,
                "num_banos": 1,
                "area": 80,
                "num_garages": 1
            }
        }
    ]
    
    predicciones = []  # Lista para almacenar las predicciones obtenidas
    for escenario in escenarios:
        # Realizamos petición POST al endpoint /predict con los datos del escenario
        response = client.post("/predict", json=escenario["datos"])
        if response.status_code == 200:  # Si la petición es exitosa
            pred = response.json()["prediction"]  # Extraemos la predicción
            predicciones.append(pred)  # Guardamos en la lista
            print(f"\n   📊 {escenario['nombre']}:")
            print(f"      Entrada: {escenario['datos']}")
            print(f"      Predicción: ${pred:,.2f}")  # Formateamos como moneda
    
    # 3. Analizar variabilidad
    print("\n" + "=" * 70)
    print("3. ANÁLISIS DE RESULTADOS")
    print("=" * 70)
    
    # Verificamos si todas las predicciones son iguales
    if len(set(predicciones)) == 1:
        print("\n   ❌ PROBLEMA DETECTADO: Todas las predicciones son IGUALES")
        print(f"      Valor constante: ${predicciones[0]:,.2f}")
        
        # Posibles causas del problema
        print("\n   🔍 POSIBLES CAUSAS:")
        print("   1. El modelo está sobreentrenado o es constante")
        print("   2. Error en el mapeo de características (ciudad vs sector)")
        print("   3. El modelo no está usando correctamente las variables")
        print("   4. Problema en el preprocesamiento de datos categóricos")
        
    else:
        print("\n   ✅ Las predicciones VARÍAN correctamente")
        print(f"      Rango: ${min(predicciones):,.2f} - ${max(predicciones):,.2f}")
        print(f"      Desviación: ${np.std(predicciones):,.2f}")  # Calculamos desviación estándar
    
    # 4. Prueba de correlación
    print("\n" + "=" * 70)
    print("4. PRUEBA DE CORRELACIÓN")
    print("=" * 70)
    
    # Probamos cómo afecta variar un solo parámetro (área) manteniendo otros constantes
    print("\n   Variando SOLO el área (manteniendo otros constantes):")
    
    areas = [50, 100, 150, 200, 250]  # Diferentes valores de área a probar
    preds_area = []  # Lista para predicciones según área
    
    # Datos base para las pruebas de variación
    base_data = {
        "provincia": "Pichincha",
        "lugar": "Quito",
        "num_dormitorios": 3,
        "num_banos": 2,
        "num_garages": 1
    }
    
    for area in areas:
        test_data = base_data.copy()  # Copiamos los datos base
        test_data["area"] = area  # Modificamos solo el área
        response = client.post("/predict", json=test_data)
        if response.status_code == 200:
            pred = response.json()["prediction"]
            preds_area.append(pred)
            print(f"      Área {area}m² → ${pred:,.2f}")
    
    # Verificamos si el área afecta la predicción
    if len(set(preds_area)) == 1:
        print("\n   ❌ El área NO afecta la predicción")
        print("      Posible causa: El modelo ignora esta variable")
    else:
        print("\n   ✅ El área SÍ afecta la predicción")
    
    # 5. Verificar el modelo directamente (VERSIÓN CORREGIDA)
    print("\n" + "=" * 70)
    print("5. VERIFICACIÓN DIRECTA DEL MODELO")
    print("=" * 70)
    
    try:
        # Determinamos la ruta base según el contexto de ejecución
        if '__file__' in globals():  # Si estamos ejecutando desde un archivo
            current_file = Path(__file__).resolve()  # Ruta absoluta del archivo actual
            base_dir = current_file.parent  # Directorio padre
        else:
            # Opción 2: Usar el directorio de trabajo actual
            base_dir = Path.cwd()
        
        # Lista de posibles ubicaciones donde podría estar el modelo
        posibles_rutas = [
            base_dir / "models" / "rental_price_model.pkl",
            base_dir.parent / "models" / "rental_price_model.pkl",
            Path("models/rental_price_model.pkl"),
            Path("../models/rental_price_model.pkl"),
            Path("../../models/rental_price_model.pkl")
        ]
        
        model_path = None
        # Buscamos el modelo en todas las rutas posibles
        for ruta in posibles_rutas:
            if ruta.exists():  # Verificamos si el archivo existe
                model_path = ruta
                break
        
        if model_path and model_path.exists():
            try:
                # Cargamos el modelo usando joblib
                model = joblib.load(model_path)
                print(f"   ✓ Modelo cargado exitosamente")
                
                # Verificamos el tipo de modelo
                model_type = type(model).__name__
                print(f"   ✓ Tipo de modelo: {model_type}")
                
                # Intentamos obtener parámetros del modelo de manera segura
                if hasattr(model, 'get_params'):
                    try:
                        params = model.get_params()  # Obtenemos los parámetros
                        print(f"   ✓ Parámetros del modelo: {params}")
                    except:
                        print("   ⚠ No se pudieron obtener los parámetros")
                
                # Para modelos de sklearn, verificamos las características esperadas
                if hasattr(model, 'feature_names_in_'):
                    try:
                        features = model.feature_names_in_  # Nombres de características
                        print(f"   ✓ Características esperadas: {features}")
                    except:
                        print("   ⚠ No se pudieron obtener los nombres de características")
                
                if hasattr(model, 'n_features_in_'):
                    try:
                        n_features = model.n_features_in_  # Número de características
                        print(f"   ✓ Número de características: {n_features}")
                    except:
                        print("   ⚠ No se pudo obtener el número de características")
                
                # Verificamos coeficientes si existen (para modelos lineales)
                if hasattr(model, 'coef_'):
                    try:
                        coef = model.coef_
                        print(f"   ✓ Coeficientes del modelo: {coef}")
                        # Verificamos si todos los coeficientes son cero
                        if hasattr(coef, 'all') and coef.all() == 0:
                            print("   ❌ ADVERTENCIA: Todos los coeficientes son cero")
                    except:
                        print("   ⚠ No se pudieron obtener los coeficientes")
                
                # Verificamos si el modelo es un pipeline
                if hasattr(model, 'steps'):
                    print(f"   ✓ El modelo es un Pipeline con {len(model.steps)} pasos")
                    for i, (name, step) in enumerate(model.steps):
                        print(f"      Paso {i+1}: {name} - {type(step).__name__}")
                
            except Exception as e:
                print(f"   ❌ Error al cargar o inspeccionar el modelo: {e}")
                print(f"   Tipo de error: {type(e).__name__}")
                traceback.print_exc()  # Imprimimos la traza completa del error
            
        else:
            print(f"\n   ❌ No se encontró el archivo del modelo")
            
    except Exception as e:
        print(f"\n   ❌ Error en verificación: {e}")
        print(f"   Tipo de error: {type(e).__name__}")
        traceback.print_exc()  # Imprimimos la traza completa del error

def prueba_original_con_datos():
    """
    Función que realiza una prueba básica con datos específicos,
    mostrando de forma detallada los datos enviados y la respuesta recibida.
    """
    print("\n" + "=" * 70)
    print("📋 PRUEBA ORIGINAL MEJORADA")
    print("=" * 70)
    
    # Datos de entrada para la prueba
    datos_entrada = {
        "provincia": "Pichincha",
        "lugar": "Quito",
        "num_dormitorios": 3,
        "num_banos": 2,
        "area": 120,
        "num_garages": 1
    }
    
    print("\n📤 DATOS ENVIADOS A LA API:")
    print(f"   Provincia: {datos_entrada['provincia']}")
    print(f"   Lugar: {datos_entrada['lugar']}")
    print(f"   Número de dormitorios: {datos_entrada['num_dormitorios']}")
    print(f"   Número de baños: {datos_entrada['num_banos']}")
    print(f"   Área: {datos_entrada['area']} m²")
    print(f"   Número de garajes: {datos_entrada['num_garages']}")
    
    # Realizamos la petición POST a la API
    response = client.post("/predict", json=datos_entrada)
    
    print("\n📥 RESPUESTA DE LA API:")
    if response.status_code == 200:  # Si la respuesta es exitosa
        resultado = response.json()
        print(f"   ✓ Código: {response.status_code}")
        print(f"   ✓ Predicción: ${resultado['prediction']:,.2f}")
        print(f"   ✓ Respuesta completa: {resultado}")
    else:
        print(f"   ✗ Código: {response.status_code}")
        print(f"   ✗ Error: {response.json()}")
    
    return response

# Punto de entrada principal del script
if __name__ == "__main__":
    # Ejecutamos primero el diagnóstico completo
    diagnosticar_modelo()
    
    # Luego ejecutamos la prueba original mejorada
    prueba_original_con_datos()
    
    print("\n" + "=" * 70)
    print("✅ DIAGNÓSTICO COMPLETADO")
    print("=" * 70)

🔍 DIAGNÓSTICO DE API - Predicción de Alquileres

1. VERIFICANDO ESTADO DE LA API...
   ✓ Health check: {'status': 'healthy', 'model_loaded': True}

2. PROBANDO DIFERENTES ESCENARIOS...

   📊 Mínimo posible:
      Entrada: {'provincia': 'Pichincha', 'lugar': 'Quito', 'num_dormitorios': 1, 'num_banos': 1, 'area': 30, 'num_garages': 0}
      Predicción: $295.76

   📊 Máximo posible:
      Entrada: {'provincia': 'Guayas', 'lugar': 'Guayaquil', 'num_dormitorios': 6, 'num_banos': 5, 'area': 500, 'num_garages': 4}
      Predicción: $855.83

   📊 Valores medios:
      Entrada: {'provincia': 'Azuay', 'lugar': 'Cuenca', 'num_dormitorios': 3, 'num_banos': 2, 'area': 150, 'num_garages': 2}
      Predicción: $601.81

   📊 Provincia diferente:
      Entrada: {'provincia': 'Manabí', 'lugar': 'Manta', 'num_dormitorios': 2, 'num_banos': 1, 'area': 80, 'num_garages': 1}
      Predicción: $424.90

3. ANÁLISIS DE RESULTADOS

   ✅ Las predicciones VARÍAN correctamente
      Rango: $295.76 - $855.83
      D